# 01. Statcast 데이터 수집
- 대상: 2021~2025 MLB 전체 선발투수
- 출처: Baseball Savant (pybaseball)
- 저장: 연도별 parquet

In [ ]:
# ── 환경 감지 ──────────────────────────────────────────────
import os

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/투수 컨디션 예측 ML'
else:
    DRIVE = os.path.dirname(os.path.abspath('__file__'))

RAW_DIR = os.path.join(DRIVE, '0_data', '1_raw')
os.makedirs(RAW_DIR, exist_ok=True)

print(f'환경: {"코랩" if IN_COLAB else "로컬"}')
print(f'RAW_DIR: {RAW_DIR}')

In [ ]:
# ── 라이브러리 설치 ────────────────────────────────────────
# 코랩은 pybaseball이 없으므로 자동 설치
try:
    import pybaseball
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'pybaseball', '-q'])
    import pybaseball

In [ ]:
import pandas as pd
import time
from pybaseball import statcast
from pybaseball import cache

# 캐시 활성화 (재실행 시 API 재호출 방지)
cache.enable()

In [ ]:
# ── 수집 설정 ──────────────────────────────────────────────
# 정규시즌 날짜 범위 (개막일~종료일)
SEASONS = {
    2021: ('2021-04-01', '2021-10-03'),
    2022: ('2022-04-07', '2022-10-05'),
    2023: ('2023-03-30', '2023-10-01'),
    2024: ('2024-03-20', '2024-09-29'),
    2025: ('2025-03-27', '2025-10-01'),  
}

print('수집 대상 시즌:')
for yr, (s, e) in SEASONS.items():
    print(f'  {yr}: {s} ~ {e}')

In [ ]:
# ── 연도별 수집 및 저장 ────────────────────────────────────
# 시간이 오래 걸림 (연도당 20~40분 예상)
# 이미 저장된 연도는 건너뜀

for year, (start, end) in SEASONS.items():
    out_path = os.path.join(BASE_DIR, f'statcast_{year}.parquet')

    if os.path.exists(out_path):
        print(f'[{year}] 이미 존재 → 건너뜀 ({out_path})')
        continue

    print(f'[{year}] 수집 시작... ({start} ~ {end})')
    t0 = time.time()

    df = statcast(start_dt=start, end_dt=end)

    elapsed = time.time() - t0
    print(f'[{year}] 수집 완료 — {len(df):,}행, {elapsed/60:.1f}분 소요')

    df.to_parquet(out_path, index=False)
    print(f'[{year}] 저장 완료 → {out_path}')
    print()

print('전체 수집 완료!')

In [ ]:
# ── 수집 결과 확인 ─────────────────────────────────────────
print('=== 저장된 파일 확인 ===')
total_rows = 0
for year in SEASONS:
    path = os.path.join(BASE_DIR, f'statcast_{year}.parquet')
    if os.path.exists(path):
        df = pd.read_parquet(path)
        size_mb = os.path.getsize(path) / 1024 / 1024
        total_rows += len(df)
        print(f'  {year}: {len(df):,}행  |  {size_mb:.1f} MB')
    else:
        print(f'  {year}: 미수집')

print(f'\n전체: {total_rows:,}행')

In [ ]:
# ── 샘플 확인 ─────────────────────────────────────────────
sample = pd.read_parquet(os.path.join(BASE_DIR, 'statcast_2024.parquet'))

# 주요 컬럼만 확인
cols = [
    'game_pk', 'game_date', 'game_type',
    'pitcher', 'player_name',
    'pitch_type', 'release_speed', 'release_spin_rate',
    'release_pos_x', 'release_pos_z', 'release_extension',
    'inning', 'inning_topbot',
    'at_bat_number', 'pitch_number',
    'description', 'events',
    'woba_value', 'woba_denom',
    'estimated_woba_using_speedangle',
    'arm_angle',
]
existing_cols = [c for c in cols if c in sample.columns]
print(f'전체 컬럼 수: {len(sample.columns)}')
print(f'\n주요 컬럼 샘플 (5행):')
sample[existing_cols].head()